# 04 · Two-tower finder

The trending finder gives everyone the same 500 articles. The two-tower finder builds a cart **for you**:

- **Item tower:** turns each article into a list of numbers (a *vector*) from its text.
- **User tower:** turns *you* into a vector from the articles you clicked before.
- Articles whose vectors are closest to yours are the ones you'd probably like.

## Step 1: Article text → numbers

We use **MiniLM**, a small pre-trained language model, *frozen*. It reads each article's category, subcategory, title and abstract and outputs 384 numbers. Because it reads text, it works for brand-new articles too, which matters because 36% of validation clicks go to articles never seen in training.


In [1]:
import sys
sys.path.insert(0, "..")

import numpy as np
import pandas as pd

from src.data import load_news
from src.finder import EMBEDDINGS_PATH, article_text, embed_articles, load_embeddings, save_embeddings

In [2]:
news = load_news()

if not EMBEDDINGS_PATH.exists():
    save_embeddings(news.news_id, embed_articles(news))

news_ids, vectors = load_embeddings()
assert (news_ids == news.news_id.values).all()
print(f"{vectors.shape[0]:,} articles × {vectors.shape[1]} numbers each")
print("Example of the text MiniLM reads:\n ", article_text(news).iloc[0])

51,282 articles × 384 numbers each
Example of the text MiniLM reads:
  lifestyle / lifestyleroyals. The Brands Queen Elizabeth, Prince Charles, and Prince Philip Swear By. Shop the notebooks, jackets, and more that the royals can't live without.


## Sanity check: Do similar articles get similar numbers?

If MiniLM works, an article's closest neighbors (by vector) should be about the same thing. The similarity score runs from 0 (unrelated) to 1 (identical).

In [3]:
def neighbors(news_id, n=5):
    i = np.flatnonzero(news_ids == news_id)[0]
    similarity = vectors @ vectors[i]
    top = np.argsort(-similarity)[: n + 1]
    return news.iloc[top][["bucket", "subcategory", "title"]].assign(similarity=similarity[top].round(2))

def find(word, subcategory):
    match = news[(news.subcategory == subcategory) & news.title.str.contains(word, case=False)]
    return match.news_id.iloc[0]

In [4]:
neighbors(find("Patriots", "football_nfl"))

,bucket,subcategory,title,similarity
99,Sports,football_nfl,"Patriots vs. Ravens: 3 keys to the game, fun f...",1.00
22839,Sports,football_nfl,Ravens vs. Patriots: Play of the week,0.85
31108,Sports,football_nfl,Mike Preston: Patriots' Bill Belichick vs. Rav...,0.84
24317,Sports,football_nfl,Patriots at Ravens: Key Matchups,0.83
10788,Sports,football_nfl,NFL Writer Explains Why Ravens Will Pull Off U...,0.82
9003,Sports,football_nfl,"Bill Belichick, not Lamar Jackson, is the smar...",0.81


In [5]:
neighbors(find("recipe", "recipes"))

,bucket,subcategory,title,similarity
34,Lifestyle,recipes,20 Must-Try Gravy Recipes for Your Thanksgivin...,1.00
31287,Lifestyle,recipes,Exceptional Gravy Recipes That Go With Everything,0.87
6153,Lifestyle,quickandeasy,45 Easy Gravy Recipes for Your Thanksgiving Feast,0.82
6171,Lifestyle,recipes,Our Favorite Thanksgiving Side Dishes Might Gi...,0.75
23491,Lifestyle,recipes,These Easy Thanksgiving Vegetables Are the New...,0.74
31206,Lifestyle,recipes,Easy Thanksgiving Salad Recipes Your Turkey Da...,0.73


In [6]:
neighbors(find("retire", "finance-retirement"))

,bucket,subcategory,title,similarity
189,Learn,finance-retirement,Want to retire and live in an RV full time? 8 ...,1.00
3557,Learn,finance-retirement,What it's really like to retire in an RV,0.76
708,Learn,finance-retirement,13 reasons you'll regret an RV in retirement,0.69
23553,Learn,finance-retirement,5 Secrets to Making Your Money Last in Retirement,0.67
17206,Learn,finance-retirement,4 things every retiree should know,0.61
29311,Lifestyle,autosenthusiasts,26 Little-Known Facts About RVs,0.60


## Do neighbors usually share a bucket?

Across 2,000 random articles: how often are the 5 nearest neighbors in the same bucket? Random would be about 20–30%. A high number means the vectors capture topic well, which the arranger and the bucket quotas rely on.

In [7]:
rng = np.random.default_rng(0)
sample = rng.choice(len(news), 2000, replace=False)
buckets = news.bucket.to_numpy(dtype=object)

similarity = vectors[sample] @ vectors.T
similarity[np.arange(len(sample)), sample] = -1  # ignore the article itself
top5 = np.argsort(-similarity, axis=1)[:, :5]

has_bucket = pd.notna(buckets[sample])
same = (buckets[top5] == buckets[sample][:, None])[has_bucket]
print(f"Nearest 5 neighbors in the same bucket: {100 * same.mean():.0f}%")
pd.Series(same.mean(axis=1), index=buckets[sample][has_bucket]).groupby(level=0).mean().mul(100).round(0).rename("same_bucket_%")

Nearest 5 neighbors in the same bucket: 82%


Entertain    60.0
Learn        71.0
Lifestyle    79.0
News         82.0
Sports       96.0
Name: same_bucket_%, dtype: float64